# Google Smartphone Decimeter Challenge 2022 — Final Submission

## Pipeline Overview

**Best public score: 2.524m** | Best private: 2.466m

```
Hatch Filter (N=1000) → Cauchy WLS → Kalman Smoother (σ=17) → 5-seed LightGBM (46 features) → SavGol (w=7)
```

### Pipeline Components
1. **Hatch carrier smoothing (N=1000):** Replaces constant-bias ADR correction with proper Hatch filter. N=1000 is converged (N=50→2.835, N=100→2.804, N=200→2.761, N=500→2.718, N=1000→2.717).
2. **Cauchy robust WLS:** Changed from `soft_l1` to `cauchy` loss in both position AND velocity `least_squares` calls. This alone dropped score from 2.717 to 2.466m.
3. **Kalman smoother:** Forward-backward RTS smoother with Mahalanobis gating at σ=17 (tested 3-30).
4. **LightGBM correction:** 5-seed ensemble (seeds 42,123,456,789,1024), min_child_samples=65, 46 extended features. Trained on σ=30 data (intentionally noisier than inference σ=17).
5. **SavGol smoothing:** window=7, polyorder=2.

### Critical: `use_tdcp=False`
All TDCP-based inference attempts scored 3.4-3.8m on private leaderboard (vs 2.466m baseline). TDCP trajectories diverge on ~4 private trips and per-trip fallback cannot detect this. **Never use TDCP for test inference.**

### Attribution & References

The core GNSS solver is adapted from **Taro Suzuki's** (Chiba Institute of Technology) public Kaggle notebook:

- **Notebook:** [Carrier Smoothing + Robust WLS + Kalman Smoother](https://www.kaggle.com/code/taroz1461/carrier-smoothing-robust-wls-kalman-smoother)
- **Paper:** Suzuki, T. (2023). "Precise Position Estimation Using Smartphone Raw GNSS Data Based on Two-Step Optimization." *Sensors*, 23(3), 1205.

**Adapted from Suzuki:** Hatch filter carrier smoothing, robust WLS solver (scipy least_squares with Sagnac correction), outlier detection thresholds (v_up > 2.6 m/s, height > 200 m), forward-backward Kalman smoother (RTS, constant-velocity model).

**Our modifications:** Cauchy loss function (replacing soft_l1), adaptive Kalman Q/R scaling by speed/HDOP, Mahalanobis gating (sigma=17), Numba JIT for Hatch filter, per-epoch feature collection.

**Phone-specific bias modelling** was inspired by **J.B.O. Mitchell's** ([jbomitchell](https://www.kaggle.com/jbomitchell)) notebooks:
- [Phone-Specific Bias Corrections (Clipped)](https://www.kaggle.com/code/jbomitchell/phone-specific-bias-corrections-clipped) — device-dependent lever arm and heading-based bias corrections
- [Tectonic Correction](https://www.kaggle.com/code/jbomitchell/tectonic-correction) — base station tectonic drift correction

Which built on **saitodevel01's** lever arm bias analysis from the 2021 competition:
- [GSDC - Bias EDA](https://www.kaggle.com/code/saitodevel01/gsdc-bias-eda)
- [GSDC - Bias Correction](https://www.kaggle.com/code/saitodevel01/gsdc-bias-correction)

**Additional references:** [Alejandro Cuevas (pollicio)](https://www.kaggle.com/pollicio) — competition notebooks that informed the overall approach.

**ML correction** (per-epoch LightGBM error prediction) was used by multiple top-10 finishers. This implementation is original.

In [ ]:
import sys, pathlib, os
import numpy as np, pandas as pd, joblib

from tqdm.notebook import tqdm
from scipy.interpolate import interp1d
from src.data_loader import load_gnss_raw
from src.gnss_solver import solve_trip_robust
from src.post_processing import median_filter_trajectory, savgol_smooth_trajectory, despike_trajectory
from src.ml_correction import PHONE_MODELS, add_rolling_features

ROOT = pathlib.Path("C:/Users/Max/PycharmProjects/sc400AGN")
MODEL_DIR = ROOT / "models"
DATASET_ROOT = ROOT / "kaggle_dataset"

## Key Parameter Choices

| Parameter | Value | Why |
|---|---|---|
| `sigma_mahalanobis` | 17.0 | Tested 3-30; σ=17 best public (2.525m). Lower rejects valid measurements, higher admits outliers |
| Ensemble seeds | 42, 123, 456, 789, 1024 | 5-seed averaging reduces variance |
| `min_child_samples` | 65 | Tuned via 2-round hyperparameter search (round 1: one-at-a-time, round 2: combinations) |
| Training σ | 30.0 | Intentionally mismatched with inference σ=17. More error variation in training → ML learns larger corrections. Matched σ=17 training scored 2.562m vs 2.525m |
| SavGol window | 7 | w=9 scored 2.830m; w=7 scored 2.804m |
| Feature count | 46 | 36 base + 10 extended. 54-feature model tested → 2.537m vs 2.525m (overfitting with 170 trips) |

In [ ]:
# Load single models (fallback)
print("Loading ML models...")
lat_model = joblib.load(MODEL_DIR / "lat_model_ext.joblib")
lon_model = joblib.load(MODEL_DIR / "lon_model_ext.joblib")
feature_cols_ext = joblib.load(MODEL_DIR / "feature_cols_ext.joblib")
print(f"  Single model: lat={lat_model.n_estimators_} trees, lon={lon_model.n_estimators_} trees")
print(f"  Features: {len(feature_cols_ext)}")

# Load 5-seed ensemble models
try:
    lat_models_ens = joblib.load(MODEL_DIR / "lat_models_ensemble.joblib")
    lon_models_ens = joblib.load(MODEL_DIR / "lon_models_ensemble.joblib")
    print(f"  Ensemble: {len(lat_models_ens)} models loaded")
    USE_ENSEMBLE = True
except FileNotFoundError:
    lat_models_ens = None
    lon_models_ens = None
    USE_ENSEMBLE = False
    print("  Ensemble models not found — using single model")

In [ ]:
# Load required output timestamps from sample submission
sample_df = pd.read_csv(DATASET_ROOT / "sample_submission.csv")
sample_ts = {}
for trip_id, grp in sample_df.groupby("tripId"):
    sample_ts[trip_id] = np.sort(grp["UnixTimeMillis"].values)
print(f"Loaded sample_submission: {len(sample_ts)} trips")

## Generate Submission

Process all 36 test trips through the full pipeline:
1. Load raw GNSS data
2. Solve with Hatch+Cauchy WLS + Kalman smoother (σ=17)
3. Median filter (kernel=3)
4. ML correction (5-seed ensemble average)
5. SavGol smoothing (w=7, poly=2)
6. Interpolate to sample submission timestamps

In [ ]:
# Configuration
SIGMA_MAHALANOBIS = 17.0
USE_ML = True
USE_SAVGOL = True

test_dir = DATASET_ROOT / "test"
all_rows = []
n_total = 0

for trip_dir in tqdm(sorted(test_dir.iterdir()), desc="Processing test trips"):
    if not trip_dir.is_dir():
        continue
    for dev_dir in sorted(trip_dir.iterdir()):
        if not dev_dir.is_dir():
            continue
        trip_id = f"{trip_dir.name}/{dev_dir.name}"
        device_name = dev_dir.name

        try:
            gnss_df = load_gnss_raw(dev_dir)
            if len(gnss_df) == 0:
                continue

            # WLS pipeline — use_tdcp=False for consistency with training data
            result, feat_df = solve_trip_robust(
                gnss_df, device_name=device_name,
                collect_features=True,
                use_tdcp=False,
                sigma_mahalanobis=SIGMA_MAHALANOBIS)

            if len(result) == 0 or feat_df is None or len(feat_df) == 0:
                continue

            pos_df = pd.DataFrame({
                "UnixTimeMillis": result["epoch_ms"].values.astype(np.int64),
                "LatitudeDegrees": result["lat"].values,
                "LongitudeDegrees": result["lon"].values,
            })
            pos_df["LatitudeDegrees"] = pos_df["LatitudeDegrees"].interpolate().ffill().bfill()
            pos_df["LongitudeDegrees"] = pos_df["LongitudeDegrees"].interpolate().ffill().bfill()
            pos_df = median_filter_trajectory(pos_df, kernel_size=3)

            # Build features for ML correction
            feat_df["lat"] = pos_df["LatitudeDegrees"].values[:len(feat_df)]
            feat_df["lon"] = pos_df["LongitudeDegrees"].values[:len(feat_df)]
            feat_df["phone_model"] = PHONE_MODELS.get(device_name, -1)
            feat_df = add_rolling_features(feat_df)

            # Apply ML correction
            if USE_ML:
                cols_present = [c for c in feature_cols_ext if c in feat_df.columns]
                X = feat_df[cols_present].fillna(0.0)
                if USE_ENSEMBLE and lat_models_ens is not None:
                    pred_lat_err = np.mean([m.predict(X) for m in lat_models_ens], axis=0)
                    pred_lon_err = np.mean([m.predict(X) for m in lon_models_ens], axis=0)
                else:
                    pred_lat_err = lat_model.predict(X)
                    pred_lon_err = lon_model.predict(X)

                n = min(len(pred_lat_err), len(pos_df))
                pred_lat_err = pred_lat_err[:n]
                pred_lon_err = pred_lon_err[:n]
                pos_df = pos_df.iloc[:n].copy()

                pos_df["LatitudeDegrees"] = pos_df["LatitudeDegrees"].values - pred_lat_err
                pos_df["LongitudeDegrees"] = pos_df["LongitudeDegrees"].values - pred_lon_err

            # SavGol smoothing
            if USE_SAVGOL:
                pos_df = savgol_smooth_trajectory(pos_df, window_length=7, polyorder=2)

            # Align to sample submission timestamps
            if trip_id in sample_ts:
                target_ts = sample_ts[trip_id]
                pred_t = pos_df["UnixTimeMillis"].values
                pred_lat = pos_df["LatitudeDegrees"].values
                pred_lon = pos_df["LongitudeDegrees"].values

                if len(pred_t) >= 2:
                    f_lat = interp1d(pred_t, pred_lat, kind="linear",
                                     bounds_error=False, fill_value=(pred_lat[0], pred_lat[-1]))
                    f_lon = interp1d(pred_t, pred_lon, kind="linear",
                                     bounds_error=False, fill_value=(pred_lon[0], pred_lon[-1]))
                    out_lat = f_lat(target_ts)
                    out_lon = f_lon(target_ts)
                else:
                    out_lat = np.full(len(target_ts), pred_lat[0] if len(pred_lat) else 0.0)
                    out_lon = np.full(len(target_ts), pred_lon[0] if len(pred_lon) else 0.0)

                for ts, la, lo in zip(target_ts, out_lat, out_lon):
                    all_rows.append({
                        "tripId": trip_id,
                        "UnixTimeMillis": int(ts),
                        "LatitudeDegrees": la,
                        "LongitudeDegrees": lo,
                    })
            else:
                for _, row in pos_df.iterrows():
                    all_rows.append({
                        "tripId": trip_id,
                        "UnixTimeMillis": int(row["UnixTimeMillis"]),
                        "LatitudeDegrees": row["LatitudeDegrees"],
                        "LongitudeDegrees": row["LongitudeDegrees"],
                    })
            n_total += 1

        except Exception as exc:
            import traceback
            print(f"\nError {trip_id}: {exc}")
            traceback.print_exc()

print(f"\nProcessed {n_total} trips")

In [ ]:
out_df = pd.DataFrame(all_rows)
out_path = ROOT / "submission_wls_ext_sg.csv"
out_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"  {len(out_df)} rows, {out_df.isnull().sum().sum()} NaN values")
print(f"  Trips: {out_df['tripId'].nunique()}")

## Local Metric Evaluation

Competition metric: **mean of (p50 + p95) / 2** per trip, averaged across all trips.

This evaluates on training trips where ground truth is available. Note: local CV metric does not always correlate with public leaderboard score (e.g., matched σ training improved CV but hurt public score).

In [ ]:
from src.main import process_trip, haversine_m
from src.data_loader import load_ground_truth

train_dir = DATASET_ROOT / "train"
trip_dirs = sorted(train_dir.iterdir())[:5]  # first 5 trips; increase for full eval

all_50p, all_95p = [], []

for trip_dir in trip_dirs:
    if not trip_dir.is_dir():
        continue
    for dev_dir in sorted(trip_dir.iterdir()):
        if not dev_dir.is_dir():
            continue
        trip_id = f"{trip_dir.name}/{dev_dir.name}"
        try:
            gt = load_ground_truth(dev_dir)
            if gt is None or len(gt) == 0:
                continue
            result = process_trip(dev_dir, train=True)

            gt_t = gt["UnixTimeMillis"].values
            pred_t = result["UnixTimeMillis"].values
            pred_lat = result["LatitudeDegrees"].values
            pred_lon = result["LongitudeDegrees"].values
            gt_lat = gt["LatitudeDegrees"].values
            gt_lon = gt["LongitudeDegrees"].values

            idx = np.searchsorted(gt_t, pred_t, side="left")
            idx = np.clip(idx, 0, len(gt_t) - 1)
            idx_prev = np.maximum(idx - 1, 0)
            closer = np.abs(gt_t[idx_prev] - pred_t) < np.abs(gt_t[idx] - pred_t)
            idx = np.where(closer, idx_prev, idx)
            errors = haversine_m(pred_lat, pred_lon, gt_lat[idx], gt_lon[idx])
            errors = errors[np.isfinite(errors)]

            p50 = float(np.percentile(errors, 50))
            p95 = float(np.percentile(errors, 95))
            kaggle_score = (p50 + p95) / 2
            print(f"{trip_id}: p50={p50:.2f} p95={p95:.2f} metric={kaggle_score:.2f}")
            all_50p.append(p50)
            all_95p.append(p95)
        except Exception as e:
            print(f"Error {trip_id}: {e}")

if all_50p:
    mean_50 = float(np.mean(all_50p))
    mean_95 = float(np.mean(all_95p))
    kaggle_agg = (mean_50 + mean_95) / 2
    print(f"\nAGGREGATE: mean_50p={mean_50:.3f}m  mean_95p={mean_95:.3f}m  score={kaggle_agg:.3f}m")

## Kaggle API Submission

- **API v2:** `api.kaggle.com/v1/competitions.CompetitionApiService/`
- **Auth:** Bearer token (set `KAGGLE_TOKEN` env var or use hardcoded fallback)
- **Daily limit:** 5 submissions/day
- **Important:** Upload raw CSV with `Content-Type: text/csv` (NOT zipped). Use `blobFileTokens` (plural) in CreateSubmission payload.

In [ ]:
import requests, json, time

TOKEN = os.environ.get("KAGGLE_TOKEN", "KGAT_ddf4d2241215d840733b60043c7dc7b2")
COMPETITION = "smartphone-decimeter-2022"
API_HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}


def list_recent_submissions(n=10):
    """Show recent submissions with scores."""
    payload = {"competitionName": COMPETITION, "page": 1, "pageSize": n}
    r = requests.post(
        "https://api.kaggle.com/v1/competitions.CompetitionApiService/ListSubmissions",
        headers=API_HEADERS, json=payload, timeout=20
    )
    if r.status_code != 200:
        print(f"ListSubmissions failed: {r.status_code} - {r.text[:200]}")
        return
    subs = r.json().get("submissions", [])
    print(f"Recent {n} submissions:")
    for s in subs[:n]:
        print(f"  {s['date'][:16]} | pub={s.get('publicScore','?'):>6} | priv={s.get('privateScore','?'):>6} | {s.get('description','')[:55]}")
    return subs


def submit_file(csv_path, message, dry_run=False):
    """Submit a CSV file to the competition."""
    csv_path = pathlib.Path(csv_path)
    if not csv_path.is_absolute():
        csv_path = ROOT / csv_path

    if not csv_path.exists():
        print(f"  ERROR: File not found: {csv_path}")
        return False

    fname = csv_path.name
    file_size = csv_path.stat().st_size
    print(f"\nSubmitting: {fname} ({file_size/1024:.0f} KB)")
    print(f"  Message: {message[:70]}")

    if dry_run:
        print("  DRY RUN - not actually submitting")
        return True

    # Step 1: Start upload
    payload = {
        "competitionName": COMPETITION,
        "fileName": fname,
        "contentLength": str(file_size),
        "lastModifiedEpochSeconds": str(int(csv_path.stat().st_mtime)),
    }
    r = requests.post(
        "https://api.kaggle.com/v1/competitions.CompetitionApiService/StartSubmissionUpload",
        headers=API_HEADERS, json=payload, timeout=20
    )
    if r.status_code != 200:
        print(f"  StartUpload failed: {r.status_code}\n  {r.text[:300]}")
        return False

    upload_data = r.json()
    create_url = upload_data.get("createUrl")
    token = upload_data.get("token")
    print(f"  Upload URL obtained")

    # Step 2: Upload raw CSV
    with open(csv_path, "rb") as f:
        csv_data = f.read()
    upload_headers = {"Content-Type": "text/csv", "Content-Length": str(len(csv_data))}
    r2 = requests.put(create_url, data=csv_data, headers=upload_headers, timeout=120)
    print(f"  File upload: {r2.status_code}")
    if r2.status_code not in (200, 201):
        print(f"  Upload response: {r2.text[:200]}")
        return False

    # Step 3: Create submission
    submit_payload = {
        "competitionName": COMPETITION,
        "blobFileTokens": token,
        "submissionDescription": message,
    }
    r3 = requests.post(
        "https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission",
        headers=API_HEADERS, json=submit_payload, timeout=30
    )
    print(f"  CreateSubmission: {r3.status_code}")
    if r3.status_code == 200:
        result = r3.json()
        print(f"  Submitted! Response: {str(result)[:200]}")
        return True
    else:
        print(f"  Failed: {r3.text[:300]}")
        return False


def check_slots():
    """Check how many submission slots remain today."""
    from datetime import datetime, timezone
    payload = {"competitionName": COMPETITION, "page": 1, "pageSize": 50}
    r = requests.post(
        "https://api.kaggle.com/v1/competitions.CompetitionApiService/ListSubmissions",
        headers=API_HEADERS, json=payload, timeout=20
    )
    if r.status_code != 200:
        return -1
    subs = r.json().get("submissions", [])
    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    today_count = sum(1 for s in subs if s.get("date", "")[:10] == today)
    print(f"Submissions today (UTC {today}): {today_count}/5")
    return max(0, 5 - today_count)

In [ ]:
check_slots()

# Uncomment to submit:
# submit_file(ROOT / "submission_wls_ext_sg.csv", "WLS+Hatch+Cauchy+Kalman(s17)+ensemble5+SavGol")

In [ ]:
list_recent_submissions(10)

## Complete Score History

| Submission | Public | Private | Notes |
|---|---|---|---|
| Hatch N=1000 + Cauchy pos+vel (baseline) | 2.466 | - | Best private score |
| Sigma=4.0 | 2.615 | 2.429 | |
| Sigma=6.0 | 2.551 | 2.452 | |
| Sigma=7.0 | 2.552 | 2.430 | |
| Sigma=8.0 (retrained) | 2.559 | 2.437 | |
| Sigma=9.0 | 2.549 | 2.440 | |
| Sigma=10.0 | 2.540 | 2.450 | |
| Sigma=12.0 | 2.535 | 2.489 | |
| Sigma=15.0 | 2.532 | 2.472 | |
| Sigma=16.0 | 2.527 | 2.476 | |
| **Sigma=17.0** | **2.525** | **2.474** | **Best single public** |
| Sigma=18.0 | 2.529 | 2.464 | |
| Sigma=20.0 | 2.579 | 2.471 | |
| Sigma=25.0 | 2.621 | - | Previous best |
| Sigma blend avg(12-18) | **2.524** | 2.473 | **Best public overall** |
| Sigma blend weighted(peak 17) | 2.524 | 2.469 | |
| Kalman sqr2 (speed_q_ref=2) | 2.588 | 2.464 | |
| Kalman sqr3 | 2.571 | 2.461 | |
| Kalman sqr10 | 2.541 | 2.476 | |
| Kalman sqr20 | 2.580 | 2.465 | |
| Kalman noQ (sqr=999) | 2.604 | 2.466 | |
| Kalman hdop1 (1.0) | 2.601 | 2.474 | |
| Kalman hdop2 (2.0) | 2.537 | 2.473 | |
| Kalman hdop3 (3.0) | 2.559 | 2.484 | |
| Kalman hdop5 (5.0) | 2.578 | 2.532 | |
| Kalman noHDOP (999) | 2.632 | 2.577 | |
| Elev-weighted WLS | 2.843 | 2.643 | Cauchy already handles outlier sats |
| 54 feat + matched sigma training | 2.562 | 2.515 | Matched training hurts |
| 54 feat + sigma=30 training | 2.537 | 2.505 | Extra features overfit |
| Q-scaling inversion | 3.090 | - | Wrong direction |
| Mahalanobis=3.0 | 2.740 | 2.472 | Too aggressive |

### Eliminated Approaches
- **TDCP inference:** ALL attempts 3.4-3.8m private (permanently eliminated)
- **L1/L5 iono-free:** 3.850m (noise amplification 2.6x)
- **RAIM:** 2.969m (cauchy already handles outliers)
- **Huber loss:** 2.982m (much worse than cauchy)
- **Bearing features:** 2.933m (hurt private)
- **MAE LightGBM objective:** Great CV, terrible generalization